# Private multilingual E5 embedding cache

Notebook ini hanya untuk dataset Kaggle **Private**. Jangan mengubah visibilitas notebook atau dataset menjadi publik. Output tidak memuat teks klinis, hanya embedding, ID, dan manifest.


In [ ]:
import os, platform, socket, subprocess, sys
print('python', sys.version)
print('platform', platform.platform())
subprocess.run(['nvidia-smi'], check=True)
try:
    socket.create_connection(('huggingface.co', 443), timeout=10).close()
    print('INTERNET_OK')
except Exception as exc:
    raise RuntimeError('Internet Kaggle harus On') from exc


In [ ]:
%pip install -q --upgrade sentence-transformers==6.1.0 huggingface-hub==1.33.0


In [ ]:
import hashlib, json, time, zipfile
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import model_info
from sentence_transformers import SentenceTransformer

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/prf_embeddings')
WORK.mkdir(parents=True, exist_ok=True)
archives = sorted(INPUT_ROOT.rglob('embedding_inputs.zip'))
if len(archives) != 1:
    raise RuntimeError(f'Diharapkan tepat satu embedding_inputs.zip, ditemukan {archives}')
with zipfile.ZipFile(archives[0]) as zf:
    zf.extractall(WORK / 'input')

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

chunks = read_jsonl(WORK / 'input/chunks.jsonl')
qa = read_jsonl(WORK / 'input/qa.jsonl')
print({'chunks': len(chunks), 'qa': len(qa), 'gpu': torch.cuda.get_device_name(0)})
assert len(chunks) == 13737 and len(qa) == 858


In [ ]:
MODEL_ID = 'intfloat/multilingual-e5-base'
REVISION = model_info(MODEL_ID).sha
model = SentenceTransformer(MODEL_ID, revision=REVISION, device='cuda')
model.max_seq_length = 512
dimension = int(model.get_sentence_embedding_dimension())
print({'model': MODEL_ID, 'revision': REVISION, 'dimension': dimension})

def encode_unique(texts, prefix, batch_size=64):
    unique = list(dict.fromkeys(texts))
    formatted = [f'{prefix}: {" ".join(text.split())}' for text in unique]
    vectors = model.encode(
        formatted, batch_size=batch_size, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype(np.float32)
    positions = {text: i for i, text in enumerate(unique)}
    expanded = np.asarray([vectors[positions[text]] for text in texts], dtype=np.float32)
    return expanded, len(unique)

started = time.perf_counter()
chunk_vectors, unique_chunks = encode_unique([r['text'] for r in chunks], 'passage')
question_vectors, unique_questions = encode_unique([r['question'] for r in qa], 'query')
answer_vectors, unique_answers = encode_unique([' '.join(r['answer_texts']) for r in qa], 'passage')
elapsed = time.perf_counter() - started
print({'elapsed_seconds': elapsed, 'unique': [unique_chunks, unique_questions, unique_answers]})


In [ ]:
for name, values, expected_rows in [
    ('chunks', chunk_vectors, len(chunks)),
    ('questions', question_vectors, len(qa)),
    ('answers', answer_vectors, len(qa)),
]:
    assert values.shape == (expected_rows, dimension), (name, values.shape)
    assert values.dtype == np.float32
    assert np.isfinite(values).all()
    assert np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=1e-5, rtol=0)

OUT = WORK / 'output'
OUT.mkdir(exist_ok=True)
np.save(OUT / 'chunks.npy', chunk_vectors)
np.save(OUT / 'questions.npy', question_vectors)
np.save(OUT / 'answers.npy', answer_vectors)
(OUT / 'ids.json').write_text(json.dumps({
    'chunk_ids': [r['chunk_id'] for r in chunks],
    'qids': [r['qid'] for r in qa],
}, ensure_ascii=False), encoding='utf-8')

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

manifest = {
    'model_id': MODEL_ID,
    'revision': REVISION,
    'device': torch.cuda.get_device_name(0),
    'elapsed_seconds': elapsed,
    'dimension': dimension,
    'rows': {'chunks': len(chunks), 'questions': len(qa), 'answers': len(qa)},
    'unique_texts': {'chunks': unique_chunks, 'questions': unique_questions, 'answers': unique_answers},
    'normalized': True,
}
for filename in ['chunks.npy', 'questions.npy', 'answers.npy', 'ids.json']:
    manifest.setdefault('sha256', {})[filename] = digest(OUT / filename)
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))


In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/e5_base_embeddings', 'zip', OUT)
print('OUTPUT_ARCHIVE', archive, Path(archive).stat().st_size)
